
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>





# Load Data Lab

In this lab, you will load data into new and existing Delta tables.

## Learning Objectives
By the end of this lab, you should be able to:
- Create an empty Delta table with a provided schema
- Insert records from an existing table into a Delta table
- Use a CTAS statement to create a Delta table from files




## Run Setup

Run the following cell to configure variables and datasets for this lesson.

In [0]:
%run ./Includes/Classroom-Setup-03.4L




## Data Overview

We will work with a sample of raw Kafka data written as JSON files. 

Each file contains all records consumed during a 5-second interval, stored with the full Kafka schema as a multiple-record JSON file. 

The schema for the table:

| field  | type | description |
| ------ | ---- | ----------- |
| key    | BINARY | The **`user_id`** field is used as the key; this is a unique alphanumeric field that corresponds to session/cookie information |
| offset | LONG | This is a unique value, monotonically increasing for each partition |
| partition | INTEGER | Our current Kafka implementation uses only 2 partitions (0 and 1) |
| timestamp | LONG    | This timestamp is recorded as milliseconds since epoch, and represents the time at which the producer appends a record to a partition |
| topic | STRING | While the Kafka service hosts multiple topics, only those records from the **`clickstream`** topic are included here |
| value | BINARY | This is the full data payload (to be discussed later), sent as JSON |


## Define Schema for Empty Delta Table
Create an empty managed Delta table named **`events_raw`** using the same schema.

In [0]:
CREATE OR REPLACE TABLE events_raw
  (key BINARY, offset BIGINT, partition INT, timestamp BIGINT, topic STRING, value BINARY);





Run the cell below to confirm the table was created correctly.

In [0]:
%python
suite = DA.tests.new("Define Schema")
expected_table = lambda: spark.table("events_raw")
suite.test_not_none(lambda: expected_table(), "Created the table \"events_raw\"")
suite.test_equals(lambda: expected_table().count(), 0, "The table should have 0 records")

suite.test_schema_field(lambda: expected_table().schema, "key", "BinaryType")
suite.test_schema_field(lambda: expected_table().schema, "offset", "LongType")
suite.test_schema_field(lambda: expected_table().schema, "partition", "IntegerType")
suite.test_schema_field(lambda: expected_table().schema, "timestamp", "LongType")
suite.test_schema_field(lambda: expected_table().schema, "topic", "StringType")
suite.test_schema_field(lambda: expected_table().schema, "value", "BinaryType")

suite.display_results()
assert suite



## Insert Raw Events Into Delta Table

Once the extracted data and Delta table are ready, insert the JSON records from the **`events_json`** table into the new **`events_raw`** Delta table.

In [0]:
INSERT INTO events_raw
SELECT * FROM events_json




Manually review the table contents to ensure data was written as expected.

In [0]:
SELECT * FROM events_raw





Run the cell below to confirm the data has been loaded correctly.

In [0]:
%python
import pyspark.sql.functions as F

suite = DA.tests.new("Validate events_raw")
expected_table = lambda: spark.table("events_raw")
suite.test_not_none(lambda: expected_table(), "Created the table \"events_raw\"")
suite.test_equals(lambda: expected_table().count(), 2252, "The table should have 2252 records")

first_five = lambda: [r["timestamp"] for r in expected_table().orderBy(F.col("timestamp").asc()).limit(5).collect()]
suite.test_sequence(first_five, [1593879303631, 1593879304224, 1593879305465, 1593879305482, 1593879305746], True, "First 5 values are correct")

last_five = lambda: [r["timestamp"] for r in expected_table().orderBy(F.col("timestamp").desc()).limit(5).collect()]
suite.test_sequence(last_five, [1593881096290, 1593881095799, 1593881093452, 1593881093394, 1593881092076], True, "Last 5 values are correct")

suite.display_results()
assert suite.passed



## Create a Delta Table From Query Results

In addition to new events data, let's also load a small lookup table that provides product details that we'll use later in the course.
Use a CTAS statement to create a managed Delta table named **`item_lookup`** that extracts data from the parquet directory provided below.

In [0]:
CREATE OR REPLACE TABLE item_lookup 
AS SELECT * FROM parquet.`${da.paths.datasets}/ecommerce/raw/item-lookup`





Run the cell below to confirm the lookup table has been loaded correctly.

In [0]:
%python
suite = DA.tests.new("Validate item_lookup")
expected_table = lambda: spark.table("item_lookup")
suite.test_not_none(lambda: expected_table(), "Created the table \"item_lookup\"")

actual_values = lambda: [r["item_id"] for r in expected_table().collect()]
expected_values = ['M_PREM_Q','M_STAN_F','M_PREM_F','M_PREM_T','M_PREM_K','P_DOWN_S','M_STAN_Q','M_STAN_K','M_STAN_T','P_FOAM_S','P_FOAM_K','P_DOWN_K']
suite.test_sequence(actual_values, expected_values, False, "Contains the 12 expected item IDs")

suite.display_results()
assert suite.passed



 
Run the following cell to delete the tables and files associated with this lesson.

In [0]:
%python
DA.cleanup()


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>